# Machine Learning Homework - Group 22
## Notebook 2 - Clean & Preprocess

**Objective**: Handle missing values and anomalous observations, integrate `clientes.csv`, encode categorical variables, create and scale features if useful

**Inputs:** `data/train_raw.csv`, `data/test_raw.csv`, `data/clientes_raw.csv` (from Notebook 1)

**Outputs:** `data/train_clean.csv`, `data/test_clean.csv`, `data/feature_lists.csv`,
`artifacts/features/*.csv`, `artifacts/logs/cleaning_log.csv`,
`artifacts/figures/nb02_missing_before_after.png`

### Every action traces back to a Notebook 1 finding

| Notebook 1 finding | Action taken here |
|---|---|
| 65 train rows have every transaction field null; test has none |Dropped from `train`|
| 1e9 placeholders in Prior Mean Purchase Value and Spend (365d) | Converted to NaN + invalid-value flag. Never clipped into a real number |
| Impossible negatives in Spend (365d), Days Since Previous Purchase, Prior Mean Purchase Gap| Converted to NaN + negative-value flag |
| `prior...` nulls are structural: 603 first-time buyers and statistics features related to that are undefined after a single purchase | Filled 0 **only for those rows**, with `no_purchase_history` / `single_purchase` flags|
|Categorical nulls (~4% of rows) carry no meaning | Imputed an explicit `Missing` level, which one-hot encodes into its own column |
| `transport_method` has 80 levels, 59 of them below 1% of rows|Levels below 1% of train grouped into `Other` before encoding |
| `clientes` attributes with <= 40 levels carry real association with | Joined on `customer_id` and 100% null features dropped|
|`train` `purchase_date` between 2015-2025, `test` 2025-2026 | `purchase_year` computed only for data quality assurance but **excluded from the model** (a coefficient on a year cannot extrapolate past the training range); `purchase_month` is kept for the model |
| 29 features with \|skew\| > 1.5 and high zero-inflation | `log1p` on the skewed non-negative columns before scaling |
|`random_noise` is a planted column with near-zero association|Dropped|

### Leakage policy
`SimpleImputer`, `StandardScaler` and `OneHotEncoder` are fitted on the training rows only and then applied to test. The fitted values (medians, means, scales, kept levels, categories) are written to `artifacts/features/`.

### Order of operations
1. Environment
2. Load and verify
3. Columns roles and drops
4. Structurally empty rows
5. Treatment for impossible values
6. Missingness mechanisms
7. Integrate `clientes`
8. Categorical cleaning
9. Numeric imputation
10. Feature engineering and treatment of outliers
11. Scaling
12. Encoding
13. Assemble and validate
14. Before vs after
15. Export preprocessed data and cleaning log
16. Handover

### Cell 1 - Environment:
Setup libraries, theme, environment, cleaning log

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
pd.set_option('display.max_columns', None)

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
ARTIFACTS_DIR = ROOT / "artifacts"
FIG_DIR = ARTIFACTS_DIR / "figures"
LOG_DIR = ARTIFACTS_DIR / "logs"
FEAT_DIR = ARTIFACTS_DIR / "features"

TARGET = "repeat_purchase_90d"
PLOT_BLUE = "#1779A8"
PLOT_ORANGE = "#C96A00"

SENTINEL_FLOOR = 1e8
RARE_LEVEL_SHARE = 0.01
SKEW_CUTOFF = 1.5
META_MAX_LEVELS = 40

CLEAN_LOG = []
def log_action(step, action, columns, rows_affected, reason):
    CLEAN_LOG.append({"step": step, "action": action, "columns": columns, "rows_affected": rows_affected, "reason": reason})

def guide(ax, text):
    ax.set_title(ax.get_title() + "\n" + text, fontsize=9)

sns.set_theme(style="whitegrid")
print("Environment ready.")
print("  data      :", DATA_DIR)
print("  artifacts :", ARTIFACTS_DIR)

### Cell 2 - Load and verify:
Read Notebook 1 raw datasets and check if everything is ready. A failure here means the notebooks are out of sync.

In [ ]:
train = pd.read_csv(DATA_DIR / "train_raw.csv")
test = pd.read_csv(DATA_DIR / "test_raw.csv")
clientes = pd.read_csv(DATA_DIR / "clientes_raw.csv")

n_train_raw, n_test_raw = len(train), len(test)

print("Shapes:")
print(" train    :", train.shape)
print(" test     :", test.shape)
print(" clientes :", clientes.shape)

print("\nIntegrity checks:")
print(" target in train only          :", TARGET in train.columns and TARGET not in test.columns)
print(" train ID unique               :", bool(train["ID"].nunique() == len(train)))
print(" test ID unique                :", bool(test["ID"].nunique() == len(test)))
extra = sorted(set(test.columns) - set(train.columns))
print(" columns only in test          :", extra if extra else "none")
print(" train/test feature names match:", sorted(set(train.columns) - {TARGET}) == sorted(set(test.columns)))
assert TARGET in train.columns and TARGET not in test.columns
assert not extra, "test carries columns train never had"
log_action("load", "read train_raw / test_raw / clientes_raw from CSV", "all",
           n_train_raw + n_test_raw, "Notebook 1 hands over raw frames as CSV")

### Cell 3 - Columns roles and drops:
Classify the variables according to their roles, as has already been done on Notebook 1.

But this time, based on exploration and analysis in Notebook 1, we remove `random_noise`, identified as having little association with the target, and retain features with at least one non-missing value and no more than 40 distinct levels, excluding fully empty columns, unstructured text features (such as `street`, `adress`) and near-unique identifiers that are less suitable for modelling.

In [ ]:
ID_COLS = ["ID", "customer_id"]
DATE_COLS = ["purchase_date"]
DROP_COLS = ["random_noise"]

numeric_all = train.select_dtypes("number").columns.tolist()
numeric_features = [c for c in numeric_all if c not in ID_COLS + [TARGET] + DROP_COLS]
categorical_features = [c for c in train.select_dtypes(include=["object", "str"]).columns if c not in DATE_COLS + DROP_COLS]

print("(train):")
print(" identifiers     :", ID_COLS)
print(" target          :", TARGET)
print(" date (as string):", DATE_COLS)
print(" dropped         :", DROP_COLS)
print(" numeric feats   :", len(numeric_features))
print(" categ feats     :", len(categorical_features), categorical_features)

train = train.drop(columns=DROP_COLS)
test = test.drop(columns=DROP_COLS)
log_action("roles", "drop planted noise column", ", ".join(DROP_COLS), 0,
           "NB01 Cell 13: near-zero association with the target")

meta_keep = [c for c in clientes.columns if c != "customer_id"
             and clientes[c].notna().any() and clientes[c].nunique() <= META_MAX_LEVELS]
meta_drop = [c for c in clientes.columns if c != "customer_id" and c not in meta_keep]
print("\nclientes features:")
print(" kept    :", meta_keep)
print(" dropped :", meta_drop)
print(f" rule    : at least one value present and at most {META_MAX_LEVELS} distinct levels")
log_action("roles", "drop non-analysable clientes attributes", ", ".join(meta_drop), 0, "free text / near-unique identifiers / fully empty (customer_type)")

### Cell 4 - Structurally empty rows:
Drop the 65 `train` rows that carry no transaction data at all (they only store id's and target features).

After the drop, we check to see if it affects the target distribution. The `test` set remains unchanged to ensure that the model predicts for all customer id's.

In [ ]:
transaction_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
empty_mask = train[transaction_cols].isna().all(axis=1)

print("Structurally empty rows:")
print(" train rows with every transaction field null:", int(empty_mask.sum()))
print(" test rows with every transaction field null :", int(test[transaction_cols].isna().all(axis=1).sum()))
print(" target distribution inside those rows       :", train.loc[empty_mask, TARGET].value_counts().to_dict())

balance_before = train[TARGET].mean()
train = train.loc[~empty_mask].reset_index(drop=True)
balance_after = train[TARGET].mean()

print("\ntrain shape after drop    :", train.shape)
print(f" positive rate before/after: {balance_before:.4f} / {balance_after:.4f}")
print(" test rows unchanged        :", len(test) == n_test_raw)

These rows were drop, reducing the `train` dataset from 6534 to 6469 observations. Since their target distribution was balanced (33 repeat purchases, 32 do not), their removal had a very little effect on the repeat purchase rate (55.92% to 55.97%). No such rows were found in the `test` dataset, which would remain unchanged any way.

### Cell 5 - Treatment for impossible values:

Add a flag column and replace impossible values, identified as negative or excessively high placeholders (sentinels), with NaN.

In [ ]:
SENTINEL_COLS = ["prior_mean_purchase_value", "spend_365d"]
NEGATIVE_IMPOSSIBLE = ["prior_mean_purchase_value", "spend_365d", "days_since_previous_purchase", "prior_mean_gap_days"]

print(f"1e9 placeholders (>= {SENTINEL_FLOOR}) -> NaN + flag:")
n_sentinel = 0
for col in SENTINEL_COLS:
    hit_tr, hit_te = train[col] >= SENTINEL_FLOOR, test[col] >= SENTINEL_FLOOR
    n_sentinel += int(hit_tr.sum()) + int(hit_te.sum())
    train["flag_sentinel_" + col] = hit_tr.astype(int)
    test["flag_sentinel_" + col] = hit_te.astype(int)
    train.loc[hit_tr, col] = np.nan
    test.loc[hit_te, col] = np.nan
    print(f"  {col:30s} train {int(hit_tr.sum()):3d} | test {int(hit_te.sum()):2d}")

print("\nImpossible negatives -> NaN + flag:")
n_negative = 0
for col in NEGATIVE_IMPOSSIBLE:
    hit_tr, hit_te = train[col] < 0, test[col] < 0
    n_negative += int(hit_tr.sum()) + int(hit_te.sum())
    train["flag_negative_" + col] = hit_tr.astype(int)
    test["flag_negative_" + col] = hit_te.astype(int)
    train.loc[hit_tr, col] = np.nan
    test.loc[hit_te, col] = np.nan
    print(f"  {col:30s} train {int(hit_tr.sum()):3d} | test {int(hit_te.sum()):2d}")

flag_cols = [c for c in train.columns if c.startswith("flag_")]
print("\nindicator columns added:", flag_cols)
print("Minimum of each affected column after the fix:")
for col in SENTINEL_COLS + NEGATIVE_IMPOSSIBLE:
    print(f"    {col:32s} min = {train[col].min():>12.2f} | max = {train[col].max():>14.2f}")
log_action("invalid values", "sentinels >= 1e8 and impossible negatives -> NaN + indicator",
           ", ".join(sorted(set(SENTINEL_COLS + NEGATIVE_IMPOSSIBLE))), n_sentinel + n_negative,
           "placeholders cannot be clipped into a real value; the event is kept as a flag")

After handling 35 excessively high values (31 in `train`, 4 in `test`) and negative values (63 in `train`, 8 in `test`). We also decided do create six columns of binary indicators to preserve information about these inconsistencies, retaining potentially useful information about their occurrence. After these corrections, we checked the affected features have valid ranges and no longer can influencing the models.

### Cell 6 - Missingness mechanisms:
Treat nulls that mean "no history" differently from nulls that mean "not recorded".

First-time and single-purchase customers are identified from `prior_purchase_count`. Exactly those rows get 0 for the history columns plus two flags. Everything still missing afterwards is ordinary missingness and goes to the imputer in cell 9.

In [ ]:
hist_tr = (train["prior_purchase_count"] == 0) | (train["prior_purchase_count"] == 1)
hist_te = (test["prior_purchase_count"] == 0) | (test["prior_purchase_count"] == 1)
train["flag_no_purchase_history"] = (train["prior_purchase_count"] == 0).astype(int)
train["flag_single_purchase"] = (train["prior_purchase_count"] == 1).astype(int)
test["flag_no_purchase_history"] = (test["prior_purchase_count"] == 0).astype(int)
test["flag_single_purchase"] = (test["prior_purchase_count"] == 1).astype(int)

HISTORY_COLS = ["days_since_previous_purchase", "prior_mean_purchase_value", "prior_std_purchase_value",
                "prior_min_purchase_value", "prior_max_purchase_value", "prior_mean_gap_days",
                "prior_std_gap_days"]

print("Missingness per column (train):")
mechanism = {}
for col in HISTORY_COLS:
    nan_rows = train[col].isna()
    explained = int((nan_rows & hist_tr).sum())
    unexplained = int(nan_rows.sum()) - explained
    mechanism[col] = "structural" if unexplained == 0 and explained > 0 else "partly structural"
    print(f"  {col:32s} NaN={ int(nan_rows.sum()):5d} | explained by no/single history={explained:5d} | other={unexplained:4d} -> { mechanism[col]}")

for col in HISTORY_COLS:
    train.loc[hist_tr, col] = train.loc[hist_tr, col].fillna(0)
    test.loc[hist_te, col] = test.loc[hist_te, col].fillna(0)

remaining = {c: int(train[c].isna().sum()) for c in HISTORY_COLS if train[c].isna().any()}
print("\nrows set to 0 by the history flags (train / test):", int(hist_tr.sum()), "/", int(hist_te.sum()))
print(f"nulls left in those columns after the structural fill: {remaining} -> those go to the median imputer in Cell 9")
log_action("missing mechanism", "structural history nulls -> 0 + flags (only for those rows)",
           ", ".join(HISTORY_COLS), int(hist_tr.sum()),
           "first-time and single-purchase customers have no history to average over")

Looking at the results above, 3 variables, `prior_std_purchase_value`, `prior_min_purchase_value` and `prior_max_purchase_value` have entirely structural missingness, that is to say, the missing values are simply due to there being no previous purchase. This pattern is consistent (although unproven) with **MAR** mechanism (Missing at random), since missingness is associated with an observed variable (`prior_purchase_count`). The remaining 4 variables are only partly explained by purchase history and their missingness mechanisms cannot be determined from this analysis alone. The potentially **MAR** values associated with these customers were replaced with 0, affecting 1048 `train` rows and 46 `test` rows. The remaining missing values will be handled through median imputation in Cell 9, preserving the distinction between missing purchase history and other missing records.

### Cell 7 - Integrate clientes:
Join customer metadata to `train` and `test` without corrupting the number of rows.

Since there is a many-to-one (N:1) relationship between purchases and customer metadata, we perform a join, which uses a left join by default, with `customer_id` as the join key. Before merging, we verify that customer IDs are unique in the metadata and check the proportion of matching customers in each dataset. An `assert` statement verifies that the number of rows remains unchanged after the join.

In [ ]:
clientes_use = clientes[["customer_id"] + meta_keep].copy()
print("Join key:")
print(f" clientes customer_id unique:", bool(clientes_use["customer_id"].nunique() == len(clientes_use)))
print(f" train customers matched    : {train['customer_id'].isin(clientes_use['customer_id']).mean() * 100:.1f}%")
print(f" test  customers matched    : {test['customer_id'].isin(clientes_use['customer_id']).mean() * 100:.1f}%")

n_tr, n_te = len(train), len(test)
train = train.set_index("customer_id").join(clientes_use.set_index("customer_id")).reset_index()
test = test.set_index("customer_id").join(clientes_use.set_index("customer_id")).reset_index()

print("\nAfter join:")
print(f" train shape: {train.shape} | test shape: {test.shape}")
assert len(train) == n_tr and len(test) == n_te, "a N:1 join must not change the number of rows"
print(" row counts unchanged -> the join is N:1")

for col in meta_keep:
    if col not in categorical_features:
        categorical_features.append(col)

print("\n unmatched (NaN) rows per joined attribute (train):")
for col in meta_keep:
    print(f"    {col:24s} NaN = {int(train[col].isna().sum()):4d} | levels = {int(train[col].nunique()):3d}")
log_action("clientes join", "left join clientes attributes on customer_id", ", ".join(meta_keep),
           n_tr + n_te, "metadata is 1:1 per customer; row counts verified unchanged")

The results confirm that `customer_id` is unique in the `clientes` dataset and that all customers in both `train` and `test` were successfully matched (100%). After the join, the datasets contain 50 and 49 columns, respectively and no changes to their number of rows, confirming a successful N:1 integration. Among the added features, `island` has the most missing values (2189) while `country_or_region` and `customer_since_year` have no missing values. These missing values will be fixed during categorical cleaning, in the next cell.

### Cell 8 - Categorical cleaning:
Make the categorical columns encodable without inventing data.

We prepare the categorical features by replacing missing values and placeholder texts with the category `"Missing"`, ensuring consistent treatment across both datasets. Next, levels representing less than 1% of training observations are grouped into `"Other"` to reduce the number of features generated by one-hot encoding. These new levels are determined using only the `train` data and then applied to the `test` set to avoid data leakage. In the end of this cell, we examine the number of categories before and after grouping, the observations affected and the number of one-hot encoded features.

In [ ]:
CAT_COLS = [c for c in categorical_features if c in train.columns]
PLACEHOLDER_TEXTS = {"nan": "Missing", "": "Missing", "-": "Missing", "?": "Missing", "none": "Missing", "unknown": "Missing"}

for col in CAT_COLS:
    train[col] = train[col].astype("string").str.strip().fillna("Missing").replace(PLACEHOLDER_TEXTS)
    test[col] = test[col].astype("string").str.strip().fillna("Missing").replace(PLACEHOLDER_TEXTS)

rare_report = []
kept_levels = {}
for col in CAT_COLS:
    share = train[col].value_counts(normalize=True)
    keep = share[share >= RARE_LEVEL_SHARE].index.tolist()
    kept_levels[col] = keep
    folded = train[col].map(lambda v, k=keep: v if v in k else "Other")
    n_folded = int((folded != train[col]).sum())
    train[col] = folded
    test[col] = test[col].map(lambda v, k=keep: v if v in k else "Other")
    rare_report.append({"feature": col, "levels_before": int(share.shape[0]), "levels_after": int(train[col].nunique()), "train_rows_grouped": n_folded,
                        "test_rows_grouped": int((test[col] == "Other").sum())})

rep = pd.DataFrame(rare_report)
print(f"Categorical levels after Missing + rare grouping (rule: keep levels >= {RARE_LEVEL_SHARE:.0%} of train):")
display(rep.set_index("feature"))
print(f"\n one-hot columns these {len(CAT_COLS)} inputs will produce (drop=first): {int((rep['levels_after'] - 1).sum())}  [purchase_month is added in Cell 10]")
print("\nLevels flagged Missing (train):")
display(pd.Series({c: int((train[c] == "Missing").sum()) for c in CAT_COLS}))
log_action("categorical", "nulls/placeholder text -> 'Missing'; levels below 1% of train -> 'Other'",
           ", ".join(CAT_COLS), int(rep["train_rows_grouped"].sum()),
           "sparse levels give unstable rates and explode the one-hot space")

We conclude that rare-category grouping reduced the number of levels in several categorical features, particularly `transport_method` (from 81 to 23 levels) and `commercial_region` (from 33 to 12). These features also had the most training observations grouped into `"Other"`, with 593 and 298, respectively. Missing values in `island` (2189), `transport_method` (196), `payment_type` (195), `warehouse` (195) and `salesperson` (194) were stored in a separate `"Missing"` category. Overall, the 10 categorical features are expected to generate 71 one-hot encoded columns, reducing dimensionality of low-frequency categories and retaining useful information for modelling.

### Cell 9 - Numeric imputation:
Fill the remaining nulls with a value estimated from training rows only.

These values are imputed using the median, `SimpleImputer(strategy="median")`, chosen for its robustness to very skewed distributions and many outliers. The imputer is fitted exclusively and applied on `train` (`imputer.fit_transform(train...)`) and then applied `test` (`imputer.transform(test...)`). We also created flags to identify imputed observations.

In [ ]:
to_impute = [c for c in numeric_features if train[c].isna().any() or test[c].isna().any()]
mask_tr = {c: train[c].isna() for c in to_impute}
mask_te = {c: test[c].isna() for c in to_impute}

missing_before = pd.DataFrame({"train NaN": [int(mask_tr[c].sum()) for c in to_impute], "test NaN": [int(mask_te[c].sum()) for c in to_impute]}, index=to_impute)
missing_before.index.name = "feature"

print("Columns imputed:")
display(missing_before)

imputer = SimpleImputer(strategy="median")
train[to_impute] = imputer.fit_transform(train[to_impute])
test[to_impute] = imputer.transform(test[to_impute])

for col in to_impute:
    train["flag_imputed_" + col] = mask_tr[col].astype(int)
    test["flag_imputed_" + col] = mask_te[col].astype(int)

medians = pd.DataFrame({"feature": to_impute, "train_median": imputer.statistics_})
medians = medians.set_index("feature")
print("\nFitted imputation values (train medians):")
display(medians.round(2))
print("\n  remaining NaN in train numerics:", int(train[numeric_features].isna().sum().sum()),
      "| in test:", int(test[numeric_features].isna().sum().sum()))
assert train[numeric_features].isna().sum().sum() == 0 and test[numeric_features].isna().sum().sum() == 0
log_action("imputation", "SimpleImputer(median) fitted on train, applied to test",
           ", ".join(to_impute), int(medians.shape[0]),
           "skewed distributions make the median the safer centre")

5 numeric features required imputation, with `prior_std_gap_days` showing the highest number of missing values (365). The imputation process successfully eliminated all remaining missing values in the numeric features of both datasets, making them suitable for the modelling stages.

### Cell 10 - Feature engineering and treatment of outliers:
Add features that the model can use and apply a method to reduce high skewness caused by outliers.

We perform feature engineering by extracting the month and year from `purchase_date`, retaining only the month as a categorical feature since the test set contains years not observed in training. We also create two numeric features to capture the average purchase value and recent purchasing frequency. Next, we identify highly skewed numeric variables and apply a `log1p` transformation to those with non-negative values, reducing skewness and influence of extreme observations. Finally, we drop the original date feature, keeping only the derived features relevant for modelling.

In [ ]:
train["purchase_month"] = "M" + train["purchase_date"].map(
    lambda s: s[5:7] if isinstance(s, str) else "NA")
test["purchase_month"] = "M" + test["purchase_date"].map(
    lambda s: s[5:7] if isinstance(s, str) else "NA")
train["purchase_year"] = train["purchase_date"].map(
    lambda s: s[0:4] if isinstance(s, str) else "NA")
test["purchase_year"] = test["purchase_date"].map(
    lambda s: s[0:4] if isinstance(s, str) else "NA")

print("Date features:")
print(" purchase_year  (QA only, excluded):", sorted(train["purchase_year"].unique()))
print(" purchase_year  in test            :", sorted(test["purchase_year"].unique()))
print(" purchase_month (kept)             :", sorted(train["purchase_month"].unique()))
print("  -> years present in test but never seen in train:",
      sorted(set(test["purchase_year"]) - set(train["purchase_year"])), "-> year is not a feature")

train["mean_ticket_365d"] = train["spend_365d"] / train["purchase_count_365d"].clip(lower=1)
test["mean_ticket_365d"] = test["spend_365d"] / test["purchase_count_365d"].clip(lower=1)
train["recent_share_30_90d"] = train["purchase_count_30d"] / (train["purchase_count_90d"] + 1)
test["recent_share_30_90d"] = test["purchase_count_30d"] / (test["purchase_count_90d"] + 1)
new_feats = ["mean_ticket_365d", "recent_share_30_90d"]
numeric_features = numeric_features + new_feats
categorical_features = categorical_features + ["purchase_month"]

skew_now = train[numeric_features].skew()
log1p_cols = [c for c in numeric_features if abs(skew_now[c]) > SKEW_CUTOFF and train[c].min() >= 0]
for col in log1p_cols:
    train[col] = np.log1p(train[col])
    test[col] = np.log1p(test[col])

print(f"\nlog1p applied to {len(log1p_cols)} of {len(numeric_features)} numeric features (|skew| > {SKEW_CUTOFF:.1f} and min >= 0):")
print("  ", log1p_cols)
print("\nEngineered ratios:")
for col in new_feats:
    print(f"  {col:20s} median = {train[col].median():>10.2f} | skew = {train[col].skew():.2f}")

train = train.drop(columns=["purchase_date"])
test = test.drop(columns=["purchase_date"])
log_action("features", "purchase_month, two ratios, log1p on skewed columns, raw date dropped",
           ", ".join(new_feats + ["purchase_month"] + log1p_cols), 0,
           "skew needs compressing; year cannot extrapolate past the train range")

The results show that the `train` data covers 2015-2025, while the test set includes 2026, supporting the decision to exclude `purchase_year` from modelling. All 12 months are represented in `train`, allowing `purchase_month` to be retained as a potential predictor feature. Two additional features were created: `mean_ticket_365d`, which captures the purchase value mean over the past year, and `recent_share_30_90d`, which measures the concentration of purchases in the most recent 30 days relative to the last 90 days. The `log1p` transformation was applied to 28 of the 30 numeric features due to their high skewness.

### Cell 11 - Scaling:
Scale the numeric features to prevent variables with larger scales from having a disproportionate influence on the models, making coefficients in Notebook 4 comparable.

We compare `StandardScaler`, `RobustScaler` and `MinMaxScaler` to assess their suitability for our numeric features, fitted exclusively on the `train` data and applied to both datasets. Flag columns created previously are left unchanged to preserve their original purpouse.

In [ ]:
flag_cols = [c for c in train.columns if c.startswith("flag_")]
scale_cols = numeric_features

raw_sd = train[scale_cols].std()
raw_iqr = train[scale_cols].quantile(0.75) - train[scale_cols].quantile(0.25)
raw_range = train[scale_cols].max() - train[scale_cols].min()
print("=== Why standardise: what the model would see without it ===")
print("  raw standard deviations run from {:.2f} to {:.0f}  ({:.0f} times apart)".format(
    float(raw_sd.min()), float(raw_sd.max()), float(raw_sd.max() / raw_sd.min())))
print("  widest column: {} (sd {:.0f}) | narrowest: {} (sd {:.2f})".format(
    raw_sd.idxmax(), float(raw_sd.max()), raw_sd.idxmin(), float(raw_sd.min())))
print("  -> one L2 penalty shared by numbers that far apart falls almost entirely on the wide")
print("     columns, and the narrow ones are effectively free.")
print("  within-column sd each taught scaler would leave behind (1.00 = penalty shared evenly):")
print("     StandardScaler {:.2f} | RobustScaler {:.2f} | MinMaxScaler {:.2f}".format(
    float((raw_sd / raw_sd).median()),
    float((raw_sd / raw_iqr.where(raw_iqr > 0, 1.0)).median()),
    float((raw_sd / raw_range).median())))
print("  RobustScaler has no usable spread on {} of the {} columns (interquartile range = 0), so it".format(
    int((raw_iqr == 0).sum()), len(scale_cols)))
print("     centres them and leaves their scale alone; MinMaxScaler is set by the two extremes, and")
print("     heavy tails are exactly what the log1p step in cell 10 was there to compress.")
print("  StandardScaler is the only one of the three that lands every column on sd 1, and it is the")
print("  only one whose coefficients can be read as effect-per-standard-deviation in notebook 04.\n")

scaler = StandardScaler()
train[scale_cols] = scaler.fit_transform(train[scale_cols])
test[scale_cols] = scaler.transform(test[scale_cols])

params = pd.DataFrame({"column": scale_cols, "train_mean": scaler.mean_, "train_scale": scaler.scale_})
print("Scaling:")
print("  numeric columns scaled :", len(scale_cols))
print("  flag columns left 0/1  :", len(flag_cols))
example_flag = flag_cols[0]
example_sd = float(train[example_flag].std())
if example_sd > 0:
    print("  if a 0/1 column WERE scaled - {}: sd {:.4f}, so one 1 would become {:.1f} and every 0 would".format(
        example_flag, example_sd, (1 - float(train[example_flag].mean())) / example_sd))
    print("  become {:.2f}. The scale of a binary column carries no information, so it is left alone.".format(
        -float(train[example_flag].mean()) / example_sd))
print("  train mean after scaling: {:.2e}  (target 0)".format(float(train[scale_cols].mean().mean())))
print("  train std  after scaling: {:.3f}  (target 1)".format(float(train[scale_cols].std().mean())))
print("  test  mean after scaling: {:.3f}  (non-zero = the splits really do differ)".format(
    float(test[scale_cols].mean().mean())))
print("\n  fitted parameters (first 12, before scaling):")
print(params.head(12).to_string(index=False, float_format=lambda v: "{:.4f}".format(v)))
assert abs(float(train[scale_cols].mean().mean())) < 1e-9
log_action("scaling", "StandardScaler fitted on train, applied to test - numeric columns only",
           ", ".join(scale_cols), 0,
           "put features on one scale so regularisation is applied evenly")

The results show differences in the original numbers with standard deviations going from 0.11 to 923.
Of the three methods looked at, `StandardScaler` was the chosen because the previous `log1p` transformations had already reduced the influence of extreme values, making the method more effective, while `RobustScaler` had 3 features with zero interquartile range and `MinMaxScaler` was still affected by to extreme observations. After scaling, the 30 numeric features have a mean approximately equal to 0 and a standard deviation of 1 in the training set, confirming successful standardization. The `test` dataset retains a non-zero mean (0.264), reflecting differences in its distribution.

### Cell 12 - Encoding:
Turn nominal levels into columns without implying any order.

We apply One-Hot Encoding to the categorical features, including the newly created `purchase_month`, converting their levels into binary features. The encoder is fitted on `train` dataset and applied to both datasets to, once again, avoid data leakage. We use `drop="first"` to remove one level per feature and reduce redundancy. `handle_unknown="ignore"` allows `test` levels that do not exist in the `train` to be encoded as all zeros.

In [ ]:
ENC_COLS = CAT_COLS + ["purchase_month"]
enc = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
train_enc = enc.fit_transform(train[ENC_COLS])
test_enc = enc.transform(test[ENC_COLS])

names = ["cat_" + c for c in enc.get_feature_names_out(ENC_COLS)]
train_cat = pd.DataFrame(train_enc, columns=names, index=train.index)
test_cat = pd.DataFrame(test_enc, columns=names, index=test.index)

print("Encoding:")
print(" categorical inputs  :\n", ENC_COLS)
print("\n one-hot columns out :\n", len(names))
print("\n categories per input:\n", dict(zip(ENC_COLS, [len(c) for c in enc.categories_])))
print("\n example names       :\n", names[:4], "...", names[-2:])


unseen = {c: int((~test[c].isin(enc.categories_[i])).sum()) for i, c in enumerate(ENC_COLS)}
print("\n  test rows whose level was never seen in train (encoded as all-zero):")
if any(unseen.values()):
    for c, n in unseen.items():
        if n:
            print(f"    {c:24s} {n:4d} rows ({n/len(test) * 100:.1f}% of test)")
else:
    print("    none")
log_action("encoding", "OneHotEncoder(drop=first, handle_unknown=ignore) fitted on train",
           ", ".join(ENC_COLS), 0, "nominal levels must not imply an ordering")

The results show that the 11 categorical features were successfully transformed into 82 binary features, 11 of them related to `purchase_month`. `transport_method` have the most features related (23) and `commercial_zone`, `warehouse`, `document_series`, and `country_or_region` having the fewest (3). No unseen levels were found in `test`, confirming that all its categorical values were represented in the `train` dataset.

### Cell 13 - Assemble and validate:
Build the matrices that Notebook 3 receives and prove they are usable.

We recreate the datasets by putting together the scaled numeric features, binary flags and one-hot encoded categorical features, while keeping the identifiers and the target, in the proper datasets. We then check that both datasets have the same feature columns, no missing values and keep unique IDs. `assert` statements make sure that these conditions are met before moving on to the modelling stage.

In [ ]:
id_cols_tr = ["ID", "customer_id"]
train_model = pd.concat([train[id_cols_tr + [TARGET]].reset_index(drop=True), train[scale_cols].reset_index(drop=True),
                         pd.DataFrame({c: train[c].values for c in flag_cols}), train_cat.reset_index(drop=True)], axis=1)
test_model = pd.concat([test[id_cols_tr].reset_index(drop=True), test[scale_cols].reset_index(drop=True),
                        pd.DataFrame({c: test[c].values for c in flag_cols}), test_cat.reset_index(drop=True)], axis=1)

feature_cols = [c for c in train_model.columns if c not in id_cols_tr + [TARGET]]
print("Model matrices:")
print(" train:", train_model.shape, "| test:", test_model.shape)
print(f" features: {len(feature_cols)} ({len(scale_cols)} numeric + {len(flag_cols)} flags + {len(names)} one-hot)")
print(" identifiers kept unscaled:", id_cols_tr)

print("\nValidation:")
print(" NaN in train features  :", int(train_model[feature_cols].isna().sum().sum()))
print(" NaN in test  features  :", int(test_model[feature_cols].isna().sum().sum()))
print(" inf in train features :", int(np.isinf(train_model[feature_cols].to_numpy()).sum()))
print(" inf in test  features :", int(np.isinf(test_model[feature_cols].to_numpy()).sum()))
print(" columns aligned        :", sorted(train_model[feature_cols].columns) == sorted(test_model[feature_cols].columns))
print(" target unchanged       :", float(train_model[TARGET].mean()))
print(" ID preserved           :", bool(train_model["ID"].nunique() == len(train_model) and test_model["ID"].nunique() == len(test_model)))
assert train_model[feature_cols].isna().sum().sum() == 0
assert test_model[feature_cols].isna().sum().sum() == 0
assert not np.isinf(train_model[feature_cols].to_numpy()).any()
assert not np.isinf(test_model[feature_cols].to_numpy()).any()
assert sorted(train_model[feature_cols].columns) == sorted(test_model[feature_cols].columns)

This notebook final datasets contain 6469 `train` observations and 986 `test` observations, with 125 predictor features each. All validation checks passed: there are no missing values. Both datasets have the same predictor features columns and all IDs are preserved and unique. The target positive rate is still around 55.97%. These results confirm that the datasets are consistent and ready for feature selection.

### Cell 14 - Before vs after:
Show the effect of this notebook in one picture.

In [ ]:
raw = pd.read_csv(DATA_DIR / "train_raw.csv")
raw = raw.loc[~raw[transaction_cols].isna().all(axis=1)]
before = raw.isna().sum()
before = before[before > 0].sort_values(ascending=False) / len(raw) * 100
shown = [c for c in before.index if c in train_model.columns]
before = before[shown]
after = train_model[shown].isna().sum() / len(train_model) * 100

long = pd.concat([pd.DataFrame({"column": before.index, "pct_missing": before.values, "stage": "before (NB01 raw)"}),
                  pd.DataFrame({"column": after.index, "pct_missing": after.values, "stage": "after (NB02 clean)"})], ignore_index=True)

fig, ax = plt.subplots(figsize=(11, 1.5 + 0.32 * len(before)))
sns.barplot(data=long, y="column", x="pct_missing", hue="stage", orient="h",
            palette=[PLOT_BLUE, PLOT_ORANGE], ax=ax)
ax.set_title(f"Missing share per column that ever had a null ({len(before)} before -> {(after > 0).sum()} after)")
ax.set_xlabel("% of train rows missing")
ax.set_ylabel("")
plt.setp(ax.get_yticklabels(), fontsize=7)
ax.legend()
guide(ax, "How to read: share of rows with a null; blue before cleaning, orange after (none left).")
plt.tight_layout()
plt.savefig(FIG_DIR / "nb02_missing_before_after.png", dpi=200)
plt.show()

print("\nSummary:")
print(" columns present in new and raw datasets with nulls before:", len(before))
print(" columns present in new and raw datasets with nulls after :", (after > 0).sum())
print(" missing cells before     : {}".format(int(raw.isna().sum().sum())))
print(" missing cells after      : {}".format(int(train_model[feature_cols].isna().sum().sum())))
print(" new flag columns added   : {}".format(len(flag_cols)))

### Cell 15 - Export preprocessed data and cleaning log:
Save the final preprocessed `train` and `test` datasets as CSV files for the following notebooks, the feature classifications, preprocessing parameters, imputation medians and categorical encoding information to document the transformations applied and the cleaning log.

In [ ]:
train_model.to_csv(DATA_DIR / "train_clean.csv", index=False)
test_model.to_csv(DATA_DIR / "test_clean.csv", index=False)

feature_lists = pd.concat([
    pd.DataFrame({"feature": scale_cols, "kind": "numeric_scaled"}),
    pd.DataFrame({"feature": flag_cols, "kind": "indicator_flag"}),
    pd.DataFrame({"feature": names, "kind": "one_hot"}),
    pd.DataFrame({"feature": ["ID", "customer_id"], "kind": ["identifier", "identifier"]}),
    pd.DataFrame({"feature": [TARGET], "kind": ["target"]}),
    pd.DataFrame({"feature": ["purchase_year", "random_noise"] + meta_drop,
                  "kind": ["excluded_qa", "excluded_noise"] + ["excluded_metadata"] * len(meta_drop)}),
], ignore_index=True)
feature_lists.to_csv(DATA_DIR / "feature_lists.csv", index=False)

params.to_csv(FEAT_DIR / "preprocessing_params.csv", index=False)
medians.rename_axis("column").to_csv(FEAT_DIR / "imputation_medians.csv")
pd.DataFrame({"column": ENC_COLS, "categories": [", ".join(map(str, c)) for c in enc.categories_]}
             ).to_csv(FEAT_DIR / "encoder_categories.csv", index=False)
pd.DataFrame([{"column": c, "kept_levels": len(k), "grouped_into_other": "yes"}
              for c, k in kept_levels.items()]).to_csv(FEAT_DIR / "categorical_levels.csv", index=False)

log_df = pd.DataFrame(CLEAN_LOG)
log_df.to_csv(LOG_DIR / "cleaning_log.csv", index=False)
print("=== Cleaning log (the full column lists are in artifacts/logs/cleaning_log.csv) ===")
display(log_df[["step", "action", "rows_affected", "reason"]])
print("\nSaved:")
print("  data/train_clean.csv, data/test_clean.csv, data/feature_lists.csv")
print("  artifacts/features/: preprocessing_params, imputation_medians, encoder_categories, categorical_levels")
print("  artifacts/logs/cleaning_log.csv, artifacts/figures/nb02_missing_before_after.png")

### Cell 16 - Handover:
Summarise what changed and what Notebook 3 receives.

In [ ]:
print("=== Notebook 2 complete ===")
print("  train rows / test rows   : {} / {}".format(len(train_model), len(test_model)))
print("  modelled features        : {} ({} numeric, {} flags, {} one-hot)".format(
    len(feature_cols), len(scale_cols), len(flag_cols), len(names)))
print("  target positive rate     : {:.4f}  (NB01: 0.5592 before the 65 empty rows were dropped)".format(
    float(train_model[TARGET].mean())))
print("  fitted on train only     : imputer, scaler, encoder")
print("  reproducibility records  : artifacts/features/*.csv and artifacts/logs/cleaning_log.csv")
print("\nNotebook 3 receives data/train_clean.csv and data/test_clean.csv and runs the feature-")
print("selection strategy: variance filter, L1-regularised logistic regression, and a decision-")
print("tree importance screen - all fitted on the train rows only.")

---

## Where this notebook leaves the homework

**Produced files**: `data/train_clean.csv` (6469 rows), `data/test_clean.csv` (986 rows), `data/feature_lists.csv`, the fitted parameters in `artifacts/features/` and the full cleaning record in `artifacts/logs/cleaning_log.csv`.

**What changed from the raw data**: 65 empty `train` rows removed; placeholders and impossible negatives turned into NaN plus flags; structural history nulls set to 0 with flags; `clientes` merged with both datasets; categorical nulls given an explicit `Missing` level and rare levels grouped into `Other`; medians fitted on `train` only; `log1p` on the skewed columns; `purchase_month` and two ratios features added; `purchase_year` computed but excluded; everything numeric standardised and everything nominal one-hot encoded.

**Next: Notebook 3 - Feature Selection**: It takes the 125 modelled columns and runs 5 stages, comparing candidate sets by measured cross-validated F1 score.